# Extração: MongoDB -> Landing Zone
Notebook `dev_amanda_extractor` com o objetivo de conectar ao banco MongoDB de origem, ler as coleções de forma parametrizada, remover campos sensíveis (senhas, tokens e embeddings) e salvar os dados brutos no formato Parquet na Landing Zone do Unity Catalog, versionados por data/hora de execução.

### Criação do volume


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS sample_mflix.landing.mongo;

### Configurando o Ambiente
Instalação do driver do MongoDB e reiniciando o contexto do Python para reconhecer as bibliotecas.

In [0]:
%pip install pymongo
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### MongoReader
**Principais funcionalidades:**
* Conexão segura utilizando `dbutils.secrets`.
* Tratamento automático de tipos nativos do MongoDB (como `ObjectId` e `ISODate`) no método `_encode`.
* Inferência de schema nativa via PySpark (`schema_of_json_agg`) para achatamento dos dados.

In [0]:
# Databricks notebook source

# COMMAND ----------
import datetime
import json
import bson
from pymongo import MongoClient
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

class MongoReader:
    SCHEMA = StructType([
        StructField("_id", StringType(), True),
        StructField("body", StringType(), True),
    ])
    def __init__(self, database: str = "sample_mflix"):
        self.spark = SparkSession.builder.getOrCreate()
        self.database = database
        self.uri = dbutils.secrets.get(scope="conn-db", key="cnn-mongodb-sampleflix")
        self.client = MongoClient(self.uri, serverSelectionTimeoutMS=15_000,
                                  socketTimeoutMS=300_000, appName="databricks-mongodb-connector")

    @staticmethod
    def _encode(o):
        if isinstance(o, bson.ObjectId):
            return str(o)
        if isinstance(o, (datetime.datetime, datetime.date)):
            return o.isoformat()
        if isinstance(o, bson.Decimal128):
            return str(o)
        if isinstance(o, bytes):
            return o.hex()
        return str(o)

    def collections(self) -> list[str]:
        return sorted(self.client[self.database].list_collection_names())

    def count(self, colecao: str, filtro: dict | None = None) -> int:
        return self.client[self.database][colecao].count_documents(filtro or {})

    def read(self, colecao: str, filtro: dict | None = None, projecao: dict | None = None,
             limite: int | None = None, batch_size: int = 5_000,
             infer: bool = False, sample_size: int = 1_000) -> DataFrame:
        cursor = self.client[self.database][colecao].find(
            filter=filtro or {}, projection=projecao, batch_size=batch_size
        )
        if limite:
            cursor = cursor.limit(limite)

        docs = [(str(d.get("_id", "")), json.dumps(d, default=self._encode, ensure_ascii=False))
                for d in cursor]
        df = self.spark.createDataFrame(docs, schema=self.SCHEMA)
        return self.expand(df, sample_size) if infer else df

    def infer_schema(self, df: DataFrame, sample_size: int = 1_000) -> str:
        return df.limit(sample_size).select(F.expr("schema_of_json_agg(body)")).first()[0]

    def expand(self, df: DataFrame, sample_size: int = 1_000) -> DataFrame:
        ddl = self.infer_schema(df, sample_size)
        # Checando tipos de esquemas
        if ddl and not any(ddl.upper().startswith(t) for t in ["STRUCT<", "ARRAY<", "MAP<"]):
            print(f"Warning: Inferred schema is primitive type '{ddl}', returning unexpanded DataFrame")
            return df
        expandido = df.select("_id", F.from_json("body", ddl).alias("doc"))
        expandido_columns = expandido.select("doc.*").columns
        colunas_doc = [c for c in expandido_columns if c != "_id"]
        return expandido.select("_id", *[f"doc.{c}" for c in colunas_doc])

    def close(self):
        self.client.close()

# COMMAND ----------
# Testando conexao primeiro com uma coleção pequena antes de ir pras grandes

reader = MongoReader()
reader.collections()

# COMMAND ----------
reader.count("users")

# COMMAND ----------
df_users = reader.read(colecao="users", infer=True)
df_users.display()

# COMMAND ----------
# Tem que sumir o password daqui

df_users_sem_senha = reader.read(colecao="users", projecao={"password": 0}, infer=True)
df_users_sem_senha.display()

# COMMAND ----------
"password" in df_users_sem_senha.columns

# COMMAND ----------
# CATALOG.sample_mflix.mongo -> troca CATALOG pelo nome certo 

path_landing = "/Volumes/sample_mflix/landing/mongo/users"
df_users_sem_senha.write.mode("overwrite").parquet(path_landing)

# COMMAND ----------
spark.read.parquet(path_landing).count()



_id,email,name,password
59b99db5cfa9a34dcd7885b8,nikolaj_coster-waldau@gameofthron.es,Jaime Lannister,$2b$12$6vz7wiwO.EI5Rilvq1zUc./9480gb1uPtXcahDxIadgyC3PS8XCUK
59b99dbacfa9a34dcd7885c2,richard_madden@gameofthron.es,Robb Stark,$2b$12$XPLvWQW7tjWc/PX9jMVRnO8w.lR6hv144ee8pc8nDsWIAWxfwxHzy
59b99dbbcfa9a34dcd7885c3,alfie_allen@gameofthron.es,Theon Greyjoy,$2b$12$x574mziridS3mEQVTbKbY.lK.ngIDyZJnTw17G7Gk6n4lnWVSrWL.
59b99dbbcfa9a34dcd7885c4,isaac_hempstead_wright@gameofthron.es,Bran Stark,$2b$12$Z7/ztVm8eWMDwTg.doS.UO7JbsbA9IbLomND1VxIZEdAN3keW6csS
59b99dbbcfa9a34dcd7885c5,jack_gleeson@gameofthron.es,Joffrey Baratheon,$2b$12$fIYLjRg5tZcTcqy30.MJLO4JRblLhsIhFwYTgUpFBbvnf9S4ONbtW
59b99dbccfa9a34dcd7885c6,rory_mccann@gameofthron.es,Sandor Clegane,$2b$12$DKIKOBMbN76vskkRyJ1AjuELigfnx.SeG8vu73k24kfMcO637bSqa
59b99db4cfa9a34dcd7885b6,sean_bean@gameofthron.es,Ned Stark,$2b$12$UREFwsRUoyF0CRqGNK0LzO0HM/jLhgUCNNIJ9RJAqMUQ74crlJ1Vu
59b99dbdcfa9a34dcd7885c7,peter_dinklage@gameofthron.es,Tyrion Lannister,$2b$12$xtHwQNXYlQzP2REobUDlzuQimjzBlXrTx1GnwP.xkfULeuuUpRxa2
59b99dbecfa9a34dcd7885c9,liam_cunningham@gameofthron.es,Davos Seaworth,$2b$12$jbgNoWG97LHNIm4axwXDz.tkFITsmw/aylIY/lZDaJRgnHZjB029e
59b99dbdcfa9a34dcd7885c8,jason_momoa@gameofthron.es,Khal Drogo,$2b$12$7tgpVkBxUqQiYFkHjZyoMuEzFU5BSI.FYkOXHu4zCRlRsa15sHQo6


_id,email,name
59b99db5cfa9a34dcd7885b8,nikolaj_coster-waldau@gameofthron.es,Jaime Lannister
59b99dbacfa9a34dcd7885c2,richard_madden@gameofthron.es,Robb Stark
59b99dbbcfa9a34dcd7885c3,alfie_allen@gameofthron.es,Theon Greyjoy
59b99dbbcfa9a34dcd7885c4,isaac_hempstead_wright@gameofthron.es,Bran Stark
59b99dbbcfa9a34dcd7885c5,jack_gleeson@gameofthron.es,Joffrey Baratheon
59b99dbccfa9a34dcd7885c6,rory_mccann@gameofthron.es,Sandor Clegane
59b99db4cfa9a34dcd7885b6,sean_bean@gameofthron.es,Ned Stark
59b99dbdcfa9a34dcd7885c7,peter_dinklage@gameofthron.es,Tyrion Lannister
59b99dbecfa9a34dcd7885c9,liam_cunningham@gameofthron.es,Davos Seaworth
59b99dbdcfa9a34dcd7885c8,jason_momoa@gameofthron.es,Khal Drogo


185

### Regras de Negócio e Segurança
Definição das coleções alvo e mapeamento das colunas que devem ser omitidas na extração por motivos de segurança (LGPD) ou performance.

In [0]:
# COMMAND ----------
exclude_fields = {
    "users": ["password"],
    "sessions": ["jwt"],
    "embedded_movies": ["plot_embedding"],
}

colecoes = ["movies", "comments", "users", "theaters", "sessions", "embedded_movies"]

### Pipeline de Extração Automatizada
Coleções configuradas aplicando as regras:
1. **Validação:** Pula coleções vazias para não quebrar o Spark.
2. **Projeção:** Remove os campos sensíveis.
3. **Versionamento:** Salva na Landing Zone em pastas com carimbo temporal (`exec=YYYYMMDD_HHMMSS`).

In [0]:
# Timestamp único da execução para versionar os arquivos na Landing
timestamp_execucao = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

for colecao in colecoes:
    # 1. Contagem antes de ler. Pular para não dar erro no infer_schema
    qtd_origem = reader.count(colecao)
    if qtd_origem == 0:
        print(colecao, "-> VAZIA (0 registros). Pulando para a próxima...\n")
        continue
        
    campos_fora = exclude_fields.get(colecao)
    projecao = {c: 0 for c in campos_fora} if campos_fora else None

    print(colecao, "-> lendo...")
    df = reader.read(colecao=colecao, projecao=projecao, infer=True)

    # Forçando o tipo String para todas as colunas
    for coluna in df.columns:
        df = df.withColumn(coluna, F.col(coluna).cast("string"))

    # 2. Versionamento
    path = f"/Volumes/sample_mflix/landing/mongo/{colecao}/exec={timestamp_execucao}"
    
    df.write.mode("overwrite").parquet(path)

    qtd_landing = spark.read.parquet(path).count()
    print(colecao, "origem:", qtd_origem, "landing:", qtd_landing, "\n")

movies -> lendo...
movies origem: 23539 landing: 23539 

comments -> lendo...
comments origem: 50315 landing: 50315 

users -> lendo...
users origem: 185 landing: 185 

theaters -> lendo...
theaters origem: 1564 landing: 1564 

sessions -> lendo...
sessions origem: 1 landing: 1 

embedded_movies -> lendo...
embedded_movies origem: 1500 landing: 1500 



### Conferência e Encerramento

In [0]:
# COMMAND ----------
# Conferência para sessions vazia ou com problema.
reader.count("sessions")

# COMMAND ----------
reader.close()